In [1]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.optimize import least_squares
# from scipy.special import gamma, gammaincc
# from scipy.integrate import quad

# # -----------------------------
# # Model definitions
# # -----------------------------

# def Schechter_diff(N0, S0, alpha, S):
#     """Differential Schechter function dN/dS"""
#     return N0 * (S / S0)**alpha * np.exp(-S / S0)

# def Schechter_cum(N0, S0, alpha, S):
#     """Cumulative Schechter using incomplete gamma function"""
#     S = np.asarray(S, dtype=float)
#     return N0 * S0 * gamma(alpha + 1) * gammaincc(alpha + 1, S / S0)

# def DPL_diff(N0, S0, alpha, beta, S):
#     """Differential Double Power Law dN/dS"""
#     return N0 / ((S / S0)**alpha + (S / S0)**beta)

# def DPL_cum(N0, S0, alpha, beta, S):
#     """Cumulative Double Power Law by integration"""
#     def integrand(s):
#         return DPL_diff(N0, S0, alpha, beta, s)
#     results = []
#     for Si in np.atleast_1d(S):
#         val, _ = quad(integrand, Si, np.inf, limit=200)
#         results.append(val)
#     return np.array(results)


# # -----------------------------
# # Residuals for fitting
# # -----------------------------

# def log_residual(params, model_func, x, y):
#     """Residuals in log space, safe against NaNs/shape mismatches"""
#     y_model = np.asarray(model_func(*params, x), dtype=float)

#     # shape check
#     if y_model.shape != y.shape:
#         return np.full_like(y, 1e6)

#     # mask invalid values
#     mask = np.isfinite(y_model) & (y_model > 0) & (y > 0)
#     if not np.any(mask):
#         return np.full_like(y, 1e6)

#     return np.log10(y_model[mask]) - np.log10(y[mask])


# # -----------------------------
# # Example data
# # -----------------------------

# # Flux bins (mJy) and cumulative counts (fake example — replace with real data!)
# S_bins_cumulative = np.array([0.19, 0.32, 0.52, 0.86, 1.43, 2.36, 3.91, 6.47])
# N_cum = np.array([12000, 8000, 5000, 3000, 1500, 800, 300, 100])

# # -----------------------------
# # Fit models
# # -----------------------------

# # Schechter cumulative fit
# guess_sch = [5e3, 1.0, -1.5]  # N0, S0, alpha
# res_sch = least_squares(log_residual, guess_sch,
#                         args=(Schechter_cum, S_bins_cumulative, N_cum))
# N0_sch, S0_sch, alpha_sch = res_sch.x

# # DPL cumulative fit
# guess_dpl = [5e3, 1.0, 1.0, 3.0]  # N0, S0, alpha, beta
# res_dpl = least_squares(log_residual, guess_dpl,
#                         args=(DPL_cum, S_bins_cumulative, N_cum))
# N0_dpl, S0_dpl, alpha_dpl, beta_dpl = res_dpl.x

# # -----------------------------
# # Evaluate models
# # -----------------------------

# S_fit = np.logspace(np.log10(S_bins_cumulative.min()),
#                     np.log10(S_bins_cumulative.max()), 200)

# N_fit_sch = Schechter_cum(N0_sch, S0_sch, alpha_sch, S_fit)
# N_fit_dpl = DPL_cum(N0_dpl, S0_dpl, alpha_dpl, beta_dpl, S_fit)

# # -----------------------------
# # Plot
# # -----------------------------

# plt.figure(figsize=(7,5))
# plt.scatter(S_bins_cumulative, N_cum, color="k", label="Data")

# plt.plot(S_fit, N_fit_sch, label=f"Schechter fit\nN0={N0_sch:.1f}, S0={S0_sch:.2f}, α={alpha_sch:.2f}")
# plt.plot(S_fit, N_fit_dpl, label=f"DPL fit\nN0={N0_dpl:.1f}, S0={S0_dpl:.2f}, α={alpha_dpl:.2f}, β={beta_dpl:.2f}")

# plt.xscale("log")
# plt.yscale("log")
# plt.xlabel("Flux density S [mJy]")
# plt.ylabel("N(>S) [deg$^{-2}$]")
# plt.legend()
# plt.tight_layout()
# plt.show()


In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.integrate import quad
from scipy.special import gammaincc, gamma

# ============================
# Example Data (replace with your actual values)
# ============================
S_mJy = np.array([0.25, 0.42, 0.69, 1.15, 1.90, 3.14, 5.19])   # midpoints for differential
dN_dS = np.array([8.0, 5.0, 4.0, 3.0, 2.0, 1.5, 1.0])          # example counts

S_bins_cumulative = np.array([0.19, 0.32, 0.52, 0.86, 1.43, 2.36, 3.91, 6.47])
N_cum = np.array([20, 15, 12, 8, 6, 4, 2, 1])                   # example cumulative counts

# ============================
# Models
# ============================

# --- Differential Schechter ---
def Schechter_diff(N0, S0, alpha, S):
    return N0 * (S / S0)**alpha * np.exp(-S / S0)

# --- Cumulative Schechter ---
def Schechter_cum(N0, S0, alpha, S):
    return N0 * S0 * gamma(alpha + 1) * gammaincc(alpha + 1, S / S0)

# --- Differential Double Power Law (DPL) ---
def DPL_diff(N0, S0, alpha, beta, S):
    return N0 / ((S / S0)**alpha + (S / S0)**beta)

# --- Cumulative Double Power Law (DPL) ---
def DPL_cum(N0, S0, alpha, beta, S):
    def integrand(sp):
        return DPL_diff(N0, S0, alpha, beta, sp)
    integral, _ = quad(integrand, S, 100, limit=200)  # cutoff at 100 mJy
    return integral

# ============================
# Residual functions
# ============================

def log_residual(params, func, x, y):
    model = np.array([func(*params, xi) for xi in x])
    mask = (model > 0) & (y > 0)   # avoid log of negative/zero
    return np.log10(model[mask]) - np.log10(y[mask])

# ============================
# Fits
# ============================

# 1. Differential Schechter
guess = [2.2, 3.9, -1.7]
res_sch_diff = least_squares(log_residual, guess,
                             args=(Schechter_diff, S_mJy, dN_dS))
popt_sch_diff = res_sch_diff.x

# 2. Differential DPL
guess = [1e3, 1.0, -1.5, -3.0]
res_dpl_diff = least_squares(log_residual, guess,
                             args=(DPL_diff, S_mJy, dN_dS))
popt_dpl_diff = res_dpl_diff.x

# 3. Cumulative Schechter
guess = [1e3, 2.0, -1.5]
res_sch_cum = least_squares(log_residual, guess,
                            args=(Schechter_cum, S_bins_cumulative, N_cum))
popt_sch_cum = res_sch_cum.x

# 4. Cumulative DPL
guess = [1e3, 1.0, -1.5, -3.0]
res_dpl_cum = least_squares(log_residual, guess,
                            args=(DPL_cum, S_bins_cumulative, N_cum))
popt_dpl_cum = res_dpl_cum.x

# ============================
# Results
# ============================
print("\n===== FIT RESULTS =====")
print("Differential Schechter: N0 = %.3g, S0 = %.3g, alpha = %.3f" %
      tuple(popt_sch_diff))
print("Differential DPL:       N0 = %.3g, S0 = %.3g, alpha = %.3f, beta = %.3f" %
      tuple(popt_dpl_diff))
print("Cumulative Schechter:   N0 = %.3g, S0 = %.3g, alpha = %.3f" %
      tuple(popt_sch_cum))
print("Cumulative DPL:         N0 = %.3g, S0 = %.3g, alpha = %.3f, beta = %.3f" %
      tuple(popt_dpl_cum))


/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (



===== FIT RESULTS =====
Differential Schechter: N0 = -1.38, S0 = 6.87, alpha = -0.567
Differential DPL:       N0 = -0.23, S0 = 63.8, alpha = 2.893, beta = 1.747
Cumulative Schechter:   N0 = 1e+03, S0 = 2, alpha = -1.500
Cumulative DPL:         N0 = 1e+03, S0 = -933, alpha = 336.929, beta = -119.703


/var/folders/p9/hcwv87bd7lxfdw1nmx4wt8d00000gn/T/ipykernel_31083/3764472163.py:30: RuntimeWarning: invalid value encountered in double_scalars
  return N0 / ((S / S0)**alpha + (S / S0)**beta)
/var/folders/p9/hcwv87bd7lxfdw1nmx4wt8d00000gn/T/ipykernel_31083/3764472163.py:36: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  integral, _ = quad(integrand, S, 100, limit=200)  # cutoff at 100 mJy
